In [1]:
%pip install pymupdf pypdf langchain_community

  Using cached pymupdf-1.27.2.2-cp310-abi3-win_amd64.whl.metadata (3.4 kB)
  Using cached pypdf-6.10.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_classic-1.0.4-py3-none-any.whl.metadata (4.8 kB)
  Using cached sqlalchemy-2.0.49-cp312-cp312-win_amd64.whl.metadata (9.8 kB)
  Using cached aiohttp-3.13.5-cp312-cp312-win_amd64.whl.metadata (8.4 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp312-cp312-win_amd64.whl.metadata (21 kB)
  Using cached multidict-6.7.1-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached propcache-0.4.1-cp312-cp312-win_amd64.whl.metadata (14 kB)
  Using cached yarl-1.23.0-cp312-cp312-win_amd64.whl.metadat

In [ ]:
def path_changer(p):
    return p.replace("\\","/")
                     

# path_changer(r'C:\Users\hoyon\OneDrive\바탕 화면\Develope\InProgress\LLM_AI_Agent\chap11\data\OneNYC_2050_Strategic_Plan.pdf')

'C:/Users/hoyon/OneDrive/바탕 화면/Develope/InProgress/LLM_AI_Agent/chap11/data/OneNYC_2050_Strategic_Plan.pdf'

In [ ]:
from langchain_community.document_loaders import PyPDFLoader


loader = PyPDFLoader('C:/Users/hoyon/OneDrive/바탕 화면/Develope/InProgress/LLM_AI_Agent/chap11/data/OneNYC_2050_Strategic_Plan.pdf')
data_nyc = loader.load()
print(data_nyc)

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


# 텍스트 데이터를 1000자로 나누고, overlap 100자로 설정
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
all_splits = text_splitter.split_documents(data_nyc)

In [ ]:
# path_changer(r'C:\Users\hoyon\OneDrive\바탕 화면\Develope\InProgress\LLM_AI_Agent\chap11\data\2040_seoul_plan.pdf')

'C:/Users/hoyon/OneDrive/바탕 화면/Develope/InProgress/LLM_AI_Agent/chap11/data/2040_seoul_plan.pdf'

In [7]:
loader_seoul = PyPDFLoader('C:/Users/hoyon/OneDrive/바탕 화면/Develope/InProgress/LLM_AI_Agent/chap11/data/2040_seoul_plan.pdf')
data_seoul = loader_seoul.load()
seoul_splits = text_splitter.split_documents(data_seoul)


In [8]:
print(len(all_splits))
all_splits.extend(seoul_splits)
print(len(all_splits))

1023
1331


In [ ]:
from langchain_openai import OpenAIEmbeddings

from dotenv import load_dotenv
import os

load_dotenv()

embedding = OpenAIEmbeddings(model="text-embedding-3-large")
v = embedding.embed_query("뉴욕의 온실가스 저감 정책은 뭐야?")
print(v)
print(len(v))

In [10]:
from langchain_chroma import Chroma
import os

persist_directory = './chroma_store'

if not os.path.exists(persist_directory):
    print("Creating new Chroma store")
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory
    )
else:
    print("Loading existing Chroma store")
    vectorstore = Chroma(
        persist_directory=persist_directory,
        embedding_function=embedding
    )
    

Creating new Chroma store


In [11]:
retreiver = vectorstore.as_retriever(k=3)
docs = retreiver.invoke("서울시의 환경 정책이 궁금해.")

for d in docs:
    print(d)
    print('-------')

page_content='제4절 기후·환경 부문1. 개요Ÿ기후변화는 21세기에 전 지구적으로 가장 위중한 영향을 미칠 것으로 예상되며, 시민 생활의 모든 측면과 연관되어 있어 향후 서울시의 적극적인 대응이 필요하다.Ÿ탄소중립 목표뿐만 아니라 미세먼지로부터 시민 건강을 지키기 위해서는 건물, 교통, 에너지 등 도시의 주요 인프라 전반의 혁신이 요구되며, 이를 위해 새로운 기술과 혁신적 제도가 필요하다. 제로에너지 건물, 친환경 차량 및 교통 인프라의 확대, 자원·에너지 순환 기반 조성으로 온실가스와 미세먼지 배출량을 획기적으로 감축해야 한다. Ÿ기후변화에 따른 폭염, 풍수해, 도심열섬현상 등 기후재난 및 극한 기후현상이 심해질 것으로 전망되어 보다 능동적인 대비가 필요하다. Ÿ한편, 환경보존과 쾌적한 도시환경을 위해 도심 곳곳 시민 모두가 누릴 수 있는 도심숲과 생활공원 등 녹색공간을 조성하고, 이를 수변 공간과 연계하여 풍부하고 지속가능한 자연환경이 확보될 수 있도록 한다.Ÿ장기적인 측면에서 시민 개개인과 기업 등 다양한 도시 내 행위자의 적극적인 협조가 필수적이며 이를 위해 중앙정부와 서울시 환경계획 담당부서와의 협력적이고 포용적인 거버넌스 체계를 구축하도록 한다.목표 전략3-12050 탄소중립 실현을 위한 도시 인프라 전환3-1-1건물 부문의 탄소배출을 감축하기 위한 친환경 기술 개발 및 적극 적용3-1-2미래 모빌리티 기술 활용과 친환경 수송 차량 및 관련 인프라 확충3-1-3에너지 전환을 위한 청정에너지 기반 구축3-1-4대기 환경을 고려한 공간계획과 배출원 관리체계 강화3-2건강한 순환도시 조성을 위한자립적인 자원순환 체계 구축3-2-1자원순환·관리 자립을 위한 분산형 폐기물처리 시설 구축3-2-2기후 행동 포용적 거버넌스 구축을 위한 시민 행동 활성화3-3사람과 자연의 공존을 위한친환경 생태도시 구축3-3-1건물 에너지 분야 효율성 개선 및 도심 속 생물 다양성 확보3-3-2지속가능한 통합 물순환 체계 구축3-4다양한 수변을 경험할 수 있는수변감성도

In [ ]:
# %pip install langchain

  Using cached langchain-1.2.15-py3-none-any.whl.metadata (5.8 kB)
  Using cached langgraph-1.1.9-py3-none-any.whl.metadata (8.0 kB)
  Using cached langgraph_checkpoint-4.0.2-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_prebuilt-1.0.10-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.3.13-py3-none-any.whl.metadata (1.6 kB)
  Using cached ormsgpack-1.12.2-cp312-cp312-win_amd64.whl.metadata (3.3 kB)
Using cached langchain-1.2.15-py3-none-any.whl (112 kB)
Using cached langgraph-1.1.9-py3-none-any.whl (173 kB)
Using cached langgraph_checkpoint-4.0.2-py3-none-any.whl (51 kB)
Using cached langgraph_prebuilt-1.0.10-py3-none-any.whl (36 kB)
Using cached langgraph_sdk-0.3.13-py3-none-any.whl (96 kB)
Using cached ormsgpack-1.12.2-cp312-cp312-win_amd64.whl (117 kB)

   ------ --------------------------------- 1/6 [langgraph-sdk]
   ------ --------------------------------- 1/6 [langgraph-sdk]
   ------ --------------------------------- 1/6 [langgraph-sdk]
   ------ 

In [13]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(model='gpt-4o-mini')

question_answering_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "사용자의 질문에 대해 아래 context에 기반하여 답변하라.:\n\n{context}",
        ),
        MessagesPlaceholder(variable_name="messages")
    ]
)

document_chain = create_stuff_documents_chain(chat, question_answering_prompt)

In [14]:
from langchain_classic.memory import ChatMessageHistory


chat_history = ChatMessageHistory()

chat_history.add_user_message("서울시의 온실가스 저감 정책에 대해 알려줘.")

answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": docs,
    }
)

chat_history.add_ai_message(answer)

print(answer)

서울시는 온실가스 저감을 위해 여러 가지 정책과 전략을 추진하고 있습니다. 주요 내용은 다음과 같습니다:

1. **탄소중립 목표 설정**: 서울시는 2005년 대비 70%의 온실가스를 감축할 목표를 세우고 다양한 분야에서 이를 실현하기 위한 노력을 기울이고 있습니다.

2. **건물 에너지 효율화**: 건물 부문에서의 탄소배출을 감축하기 위해 친환경 기술을 개발하고 이를 적극적으로 적용하는 방안을 추진하고 있습니다. 예를 들어, 제로에너지 건물과 녹화 기법 도입을 통해 에너지 효율성을 높이는 것을 목표로 합니다.

3. **미래 모빌리티와 친환경 교통**: 친환경 차량과 관련 인프라를 확충하고, 지속가능한 교통 체계를 구축하여 교통 부문에서의 탄소 배출을 줄이고 있습니다. 

4. **청정에너지 기반 구축**: 에너지 전환을 위해 신재생에너지의 보급률을 높이고, 청정에너지를 활용하는 제도를 마련하고 있습니다. 2030년까지 21%의 신재생에너지 비율을 목표로 하고 있습니다.

5. **자원순환 체계 강화**: 자립적인 자원순환 체계를 구축하여 폐기물 발생을 줄이고 재활용률을 최대화하기 위한 정책을 운영합니다. 이를 통해 온실가스 발생을 원천적으로 감축하고 있습니다.

6. **국민 참여 및 협력**: 시민, 기업, 정부 간 협력 체계를 마련하여 기후 행동을 활성화하고, 자원순환 및 친환경 교육을 통해 시민의 감수성을 높이고 있습니다.

7. **대기 환경 관리**: 대기오염물질인 PM2.5, NOx 등의 배출을 감축하기 위해 배출원 관리체계를 강화하고, 자연적인 대기순환을 고려한 공간계획을 수립하여 대기 환경 개선을 추진하고 있습니다.

이와 같은 서울시의 정책들은 기후변화에 대응하고 지속 가능한 발전을 이루기 위한 종합적인 노력의 일환으로 진행되고 있습니다.


In [15]:
from langchain_core.output_parsers import StrOutputParser

In [17]:
query_for_nyc = "뉴욕은?"
query_argmentation_prompt = ChatPromptTemplate.from_messages(
    [
        MessagesPlaceholder(variable_name="messages"),
        (
            "system",
            "기존의 대화 내용을 활용하여 사용자가 질문한 의도를 파악해서 한 문장의 명료한 질문으로 변환하라. 대명사나 이, 저, 그와 같은 표현을 명확한 명사로 표현하라. :\n\n{query}",
        ),
    ]
)

In [18]:
query_argmentation_chain = query_argmentation_prompt | chat | StrOutputParser()